In [ ]:
from nuscenes.nuscenes import NuScenes
from ultralytics import YOLO, SAM
import torch
import numpy as np
import cv2

In [ ]:
det_model = YOLO('./ckpts/yolo11l-seg.pt')
sam_model = SAM('./ckpts/sam2.1_l.pt')

In [ ]:
nusc = NuScenes(version='v1.0-trainval-select', dataroot='../data/nuscenes', verbose=True)
sample_token = nusc.scene[3]['first_sample_token']
sample = nusc.get('sample', sample_token)
# cams = ['CAM_FRONT', 'CAM_FRONT_RIGHT', 'CAM_BACK_RIGHT', 'CAM_BACK', 'CAM_BACK_LEFT', 'CAM_FRONT_LEFT']
cams = ['CAM_FRONT']
cam_token_list =  [sample['data'][cam] for cam in cams]

In [ ]:
fps = 10
w, h = 1600, 900
video_writer_list =  []
for cam in cams:
    video_writer_list.append(cv2.VideoWriter(f"isegment_output_{cam}.mp4", cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h)))
# cv2.VideoWriter("isegment_output.mp4", cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))

In [ ]:
count = 0
while cam_token_list[0] != '':
    data_path_list = []
    for i in range(len(cams)):
        cam = nusc.get('sample_data', cam_token_list[i])
        data_path_list.append('../data/nuscenes/' + cam['filename'])
        cam_token_list[i] = cam['next']

    det_results = det_model.track(data_path_list, persist=True)
    for i in range(len(cams)):
        # class_ids = det_results[i].boxes.cls.int().tolist()
        # if class_ids:
        #     boxes = det_results[i].boxes.xyxy  # Boxes object for bbox outputs
        #     sam_results = sam_model(det_results[i].orig_img, bboxes=boxes, labels=class_ids, save=False)
        #     det_results[i].masks = sam_results[0].masks
        video_writer_list[i].write(det_results[i].plot(line_width=2, font_size=2, conf=False))

    
    # class_ids = det_results[0].boxes.cls.int().tolist()  # Extract class IDs from detection results
    # if class_ids:
    #     boxes = det_results[0].boxes.xyxy  # Boxes object for bbox outputs
    #     sam_results = sam_model(det_results[0].orig_img, bboxes=boxes, labels=class_ids, save=False)
    # det_results[0].masks = sam_results[0].masks
    # video_writer.write(det_results[0].plot(line_width=2, font_size=2, conf=False))

    # count += 1
    # if count >= 20:
    #     break

for video_writer in video_writer_list:
    video_writer.release()

In [ ]:
video_writer.release()

In [ ]:
for video_writer in video_writer_list:
    video_writer.release()